# Pipeline batch de vendas — Apache Spark e Parquet
## Medallion architecture: bronze → silver → gold
Notebook didático completo em Python/PySpark, sem Delta Lake. Volume padrão: **1.000.000 de vendas, 250.000 clientes e 100.000 produtos**. Os dados são sintéticos e determinísticos.

**Resultado:** arquivos de origem em CSV, bronze com rastreabilidade, silver tipada e validada, quarentena com motivos e gold com indicadores comerciais. Execute as células em ordem.

**Limite arquitetural:** Parquet fornece armazenamento colunar, mas não fornece transações ACID de tabela, MERGE, time travel ou publicação atômica de várias tabelas. Este exemplo implementa a organização medallion e a consulta analítica de um lakehouse simplificado. Para essas garantias, seria necessário um formato de tabela como Apache Iceberg ou Hudi, com dependências adicionais. Não há leitura ou escrita no formato Delta neste notebook.

**Validação do notebook:** executado integralmente com Python 3.12, Java 17 e PySpark 3.5.7 no volume padrão. Todas as verificações passaram: 995.000 vendas válidas e 5.000 em quarentena. Receita líquida sintética reconciliada: R$ 1.393.646.037,50. Tempo de execução depende do ambiente. As saídas foram limpas para facilitar o uso em aula.

## 1. Ambiente e instalação
Referência reproduzível: Python 3.10–3.12, Java 17 e PySpark 3.5.7. Use Jupyter local em Linux, macOS ou Windows com Java configurado. Sugestão inicial: 8 GB de RAM e alguns GB livres em disco; tempo e consumo dependem do computador.

Execute a instalação abaixo **somente se necessário**, removendo o comentário. Reinicie o kernel após instalar. Não instale `delta-spark`. Em ambientes com Spark gerenciado, use o PySpark fornecido pela plataforma.

Este notebook utiliza caminhos locais: em cluster, substitua-os por armazenamento compartilhado acessível a todos os executores. Não usa pandas nem coleta o conjunto completo para o driver.

In [1]:
# %pip install pyspark==3.5.7
import os
import sys
import subprocess
print('Python:', sys.version)
print(subprocess.run(['java', '-version'], capture_output=True, text=True).stderr)

Python: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 11:57:02) [GCC 12.3.0]
openjdk version "17.0.8.1" 2023-08-24
OpenJDK Runtime Environment (build 17.0.8.1+1-Ubuntu-0ubuntu122.04)
OpenJDK 64-Bit Server VM (build 17.0.8.1+1-Ubuntu-0ubuntu122.04, mixed mode, sharing)



## 2. Configuração e isolamento de execução
Cada execução usa uma pasta própria. Uma repetição completa cria outro snapshot, evitando misturar lotes. Dentro do snapshot, `overwrite` torna o reprocessamento da mesma etapa previsível, mas não transacional. Não execute dois escritores na mesma pasta.

`MODO_RAPIDO=True` permite ensaiar com dados menores. O padrão continua sendo o volume solicitado. O catálogo de views é temporário; os arquivos Parquet persistem.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import time
from decimal import Decimal
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.storagelevel import StorageLevel

MODO_RAPIDO = False
N_CLIENTES, N_PRODUTOS, N_VENDAS = (2500, 1000, 10000) if MODO_RAPIDO else (250000, 100000, 1000000)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
BASE = Path.cwd() / 'lakehouse_vendas_batch' / RUN_ID
BASE.mkdir(parents=True, exist_ok=False)
LANDING, BRONZE, SILVER, GOLD = [str(BASE / x) for x in ['landing','bronze','silver','gold']]
PARTICOES = 8
spark = (SparkSession.builder.master('local[4]').appName('VendasBatchParquet')
    .config('spark.sql.shuffle.partitions', str(PARTICOES))
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark:', spark.version, '| Execução:', RUN_ID, '| Pasta:', BASE)
inicio = time.perf_counter()
metricas = []

Spark: 3.5.0 | Execução: 20261011T210239023684Z | Pasta: /home/jovyan/work/lakehouse_vendas_batch/20261011T210239023684Z


## 3. Gerar o sistema de origem
`spark.range` distribui a geração; expressões nativas evitam loops Python e UDFs. Cada linha de vendas representa uma venda com um produto, portanto ticket médio = receita / quantidade de vendas.

Datas cobrem 2025. Quantidade, preço e desconto determinísticos permitem reproduzir os resultados. Injetamos problemas em subconjuntos disjuntos: quantidade zero, preço negativo, cliente inexistente, produto inexistente e desconto acima de 100%. O volume de vendas inclui essas linhas inválidas e permanece exatamente o solicitado. Dimensões não possuem defeitos neste cenário.

In [3]:
clientes = spark.range(1, N_CLIENTES + 1).select(
    F.col('id').alias('cliente_id'), F.concat(F.lit('Cliente '), F.col('id')).alias('nome'),
    F.concat(F.lit('cliente'), F.col('id'), F.lit('@exemplo.invalid')).alias('email'),
    F.element_at(F.array(*[F.lit(x) for x in ['SP','RJ','MG','PR','BA']]), ((F.col('id') % 5)+1).cast('int')).alias('uf'))
produtos = spark.range(1, N_PRODUTOS + 1).select(
    F.col('id').alias('produto_id'), F.concat(F.lit('Produto '), F.col('id')).alias('nome_produto'),
    F.element_at(F.array(*[F.lit(x) for x in ['Eletronicos','Casa','Moda','Esporte','Livros']]), ((F.col('id') % 5)+1).cast('int')).alias('categoria'))
vendas = spark.range(1, N_VENDAS + 1).select(
    F.col('id').alias('venda_id'),
    ((F.col('id') * 17 % N_CLIENTES)+1).alias('cliente_id'),
    ((F.col('id') * 13 % N_PRODUTOS)+1).alias('produto_id'),
    F.date_add(F.lit('2025-01-01').cast('date'), (F.col('id') % 365).cast('int')).alias('data_venda'),
    ((F.col('id') % 5)+1).cast('int').alias('quantidade'),
    (((F.col('id') * 37 % 99000)+1000)/100).cast('decimal(12,2)').alias('preco_unitario'),
    ((F.col('id') % 4)/20).cast('decimal(5,4)').alias('desconto_pct'),
    F.when(F.col('id') % 2 == 0, 'Online').otherwise('Loja').alias('canal'))
for campo, resto, valor in [('quantidade',0,0),('preco_unitario',1,-10),
                            ('cliente_id',2,N_CLIENTES+1),('produto_id',3,N_PRODUTOS+1),('desconto_pct',4,1.5)]:
    vendas = vendas.withColumn(campo, F.when(F.col('venda_id') % 1000 == resto, F.lit(valor)).otherwise(F.col(campo)))
for nome, df in [('clientes',clientes),('produtos',produtos),('vendas',vendas)]:
    df.repartition(PARTICOES).write.mode('overwrite').option('header',True).csv(f'{LANDING}/{nome}')
print('Origem CSV gerada.')

Origem CSV gerada.


## 4. Bronze: ingestão batch e rastreabilidade
A origem CSV é lida com esquema explícito de strings, sem inferência. Bronze preserva o conteúdo e acrescenta arquivo de origem, lote e horário de ingestão. Parquet usa compressão Snappy.

A geração da origem é uma simulação; em uso real, substitua a landing por arquivos recebidos do ERP/CRM. O lote deste exemplo é um snapshot completo, não uma implementação incremental de CDC.

In [4]:
colunas = {
 'clientes':['cliente_id','nome','email','uf'],
 'produtos':['produto_id','nome_produto','categoria'],
 'vendas':['venda_id','cliente_id','produto_id','data_venda','quantidade','preco_unitario','desconto_pct','canal']}
bronze = {}
for nome, campos in colunas.items():
    schema = T.StructType([T.StructField(c,T.StringType(),True) for c in campos])
    df = (spark.read.schema(schema).option('header',True).option('mode','FAILFAST').csv(f'{LANDING}/{nome}')
        .withColumn('_arquivo_origem',F.input_file_name())
        .withColumn('_lote',F.lit(RUN_ID)).withColumn('_ingerido_em',F.current_timestamp()))
    df.write.mode('overwrite').option('compression','snappy').parquet(f'{BRONZE}/{nome}')
    bronze[nome] = spark.read.parquet(f'{BRONZE}/{nome}')
    n = bronze[nome].count()
    assert n == {'clientes':N_CLIENTES,'produtos':N_PRODUTOS,'vendas':N_VENDAS}[nome]
    metricas.append((nome,'bronze',n))
bronze['vendas'].show(5,truncate=False)

+--------+----------+----------+----------+----------+--------------+------------+------+----------------------------------------------------------------------------------------------------------------------------------------------+----------------------+-------------------------+
|venda_id|cliente_id|produto_id|data_venda|quantidade|preco_unitario|desconto_pct|canal |_arquivo_origem                                                                                                                               |_lote                 |_ingerido_em             |
+--------+----------+----------+----------+----------+--------------+------------+------+----------------------------------------------------------------------------------------------------------------------------------------------+----------------------+-------------------------+
|219647  |234000    |55412     |2025-10-10|3         |99.39         |0.15        |Loja  |file:///home/jovyan/work/lakehouse_vendas_batch/20261011T21023902

## 5. Silver: dimensões e contrato de dados
Convertemos IDs, padronizamos texto e validamos chaves antes dos joins. Dimensões inválidas interrompem o lote para não multiplicar vendas nem esconder defeitos. `try_cast` transforma conversões inválidas em NULL mesmo com modo ANSI habilitado.

Não usamos `dropDuplicates` indiscriminadamente: duplicidades de chave interrompem a execução, pois escolher um registro sem regra de prioridade seria arbitrário.

In [5]:
sc = bronze['clientes'].selectExpr('try_cast(cliente_id as bigint) cliente_id','trim(nome) nome','lower(trim(email)) email','upper(trim(uf)) uf')
sp = bronze['produtos'].selectExpr('try_cast(produto_id as bigint) produto_id','trim(nome_produto) nome_produto','trim(categoria) categoria')
for nome, df, chave, campos, esperado in [
    ('clientes',sc,'cliente_id',['nome','email','uf'],N_CLIENTES),
    ('produtos',sp,'produto_id',['nome_produto','categoria'],N_PRODUTOS)]:
    ruim = F.col(chave).isNull() | (F.col(chave) <= 0)
    for c in campos:
        ruim = ruim | F.col(c).isNull() | (F.length(F.col(c)) == 0)
    assert df.filter(ruim).limit(1).count() == 0, f'Dimensão inválida: {nome}'
    assert df.groupBy(chave).count().filter('count > 1').limit(1).count() == 0, f'Chave duplicada: {nome}'
    assert df.count() == esperado
    df.write.mode('overwrite').parquet(f'{SILVER}/{nome}')
sc, sp = spark.read.parquet(f'{SILVER}/clientes'), spark.read.parquet(f'{SILVER}/produtos')

## 6. Silver: vendas e quarentena
Cada registro recebe uma lista de motivos. Validamos obrigatoriedade, domínios, datas e integridade referencial. Uma linha com vários problemas aparece uma única vez na quarentena, com todos os motivos. IDs de venda duplicados interrompem o lote.

A quarentena mantém valores originais para permitir diagnóstico de erros de conversão. Valores monetários usam Decimal; receita líquida considera desconto, sem impostos, frete ou devoluções.

In [6]:
tipos = {'venda_id':'bigint','cliente_id':'bigint','produto_id':'bigint','data_venda':'date',
         'quantidade':'int','preco_unitario':'decimal(12,2)','desconto_pct':'decimal(5,4)'}
sv = bronze['vendas']
for c, tipo in tipos.items():
    sv = sv.withColumn(f'_original_{c}',F.col(c)).withColumn(c,F.expr(f'try_cast({c} as {tipo})'))
sv = sv.withColumn('canal',F.trim('canal'))
assert sv.filter('venda_id is not null').groupBy('venda_id').count().filter('count > 1').limit(1).count() == 0, 'Venda duplicada'
sv = (sv.join(sc.select('cliente_id').withColumn('_cliente_existe',F.lit(True)), 'cliente_id','left')
        .join(sp.select('produto_id').withColumn('_produto_existe',F.lit(True)), 'produto_id','left'))
regras = [
 ('venda_id_invalido', F.col('venda_id').isNull() | (F.col('venda_id') <= 0)),
 ('cliente_inexistente',F.col('_cliente_existe').isNull()),
 ('produto_inexistente',F.col('_produto_existe').isNull()),
 ('data_invalida', F.col('data_venda').isNull() | ~F.col('data_venda').between('2025-01-01','2025-12-31')),
 ('quantidade_invalida',F.col('quantidade').isNull() | (F.col('quantidade') <= 0)),
 ('preco_invalido',F.col('preco_unitario').isNull() | (F.col('preco_unitario') <= 0)),
 ('desconto_invalido',F.col('desconto_pct').isNull() | ~F.col('desconto_pct').between(0,1)),
 ('canal_invalido',F.col('canal').isNull() | ~F.col('canal').isin('Online','Loja'))]
classificadas = sv.withColumn('motivos', F.filter(F.array(*[
    F.when(cond,F.lit(motivo)) for motivo,cond in regras]),lambda x:x.isNotNull())).persist(StorageLevel.MEMORY_AND_DISK)
quarentena = classificadas.filter(F.size('motivos') > 0)
validas = (classificadas.filter(F.size('motivos') == 0)
    .select(*colunas['vendas'],'_lote','_ingerido_em','_arquivo_origem')
    .withColumn('valor_bruto',(F.col('quantidade')*F.col('preco_unitario')).cast('decimal(18,2)')))
validas = (validas.withColumn('valor_desconto',(F.col('valor_bruto')*F.col('desconto_pct')).cast('decimal(18,2)'))
    .withColumn('valor_liquido',(F.col('valor_bruto')-F.col('valor_desconto')).cast('decimal(18,2)'))
    .withColumn('ano',F.year('data_venda')).withColumn('mes',F.month('data_venda')))
n_validas, n_quarentena = validas.count(), quarentena.count()
assert n_validas + n_quarentena == N_VENDAS
assert n_quarentena == (N_VENDAS // 1000) * 5
validas.repartition(PARTICOES,'mes').write.mode('overwrite').partitionBy('ano','mes').parquet(f'{SILVER}/vendas')
quarentena.write.mode('overwrite').parquet(f'{SILVER}/quarentena_vendas')
metricas.extend([('vendas','silver',n_validas),('quarentena_vendas','silver',n_quarentena)])
quarentena.select(F.explode('motivos').alias('motivo')).groupBy('motivo').count().orderBy('motivo').show()
classificadas.unpersist()
v = spark.read.parquet(f'{SILVER}/vendas')
print(f'Vendas válidas: {n_validas:,} | Quarentena: {n_quarentena:,}')

+-------------------+-----+
|             motivo|count|
+-------------------+-----+
|cliente_inexistente| 1000|
|  desconto_invalido| 1000|
|     preco_invalido| 1000|
|produto_inexistente| 1000|
|quantidade_invalida| 1000|
+-------------------+-----+

Vendas válidas: 995,000 | Quarentena: 5,000


## 7. Gold: modelo analítico e indicadores
A fato enriquecida mantém a granularidade de uma venda; joins com dimensões de chave única preservam sua contagem. As agregações são calculadas com Spark SQL, invocado pelo Python.

Produzimos receita diária por canal, receita mensal por categoria e UF, ranking de produtos e resumo executivo. Clientes distintos não devem ser somados entre grupos, pois um cliente pode aparecer em vários grupos. O ranking usa desempate por ID.

In [7]:
fato = (v.join(sc,'cliente_id').join(sp,'produto_id')
    .select('venda_id','cliente_id','produto_id','data_venda','quantidade','canal','uf','categoria',
            'nome_produto','valor_bruto','valor_desconto','valor_liquido','ano','mes'))
assert fato.count() == n_validas
fato.createOrReplaceTempView('fato_vendas')
consultas = {
 'vendas_diarias': """SELECT data_venda, canal, COUNT(*) vendas, SUM(quantidade) unidades,
    SUM(valor_bruto) receita_bruta, SUM(valor_desconto) descontos, SUM(valor_liquido) receita_liquida,
    CAST(SUM(valor_liquido)/COUNT(*) AS DECIMAL(18,2)) ticket_medio,
    COUNT(DISTINCT cliente_id) clientes_distintos FROM fato_vendas GROUP BY data_venda, canal""",
 'vendas_mensais': """SELECT ano, mes, categoria, uf, COUNT(*) vendas,
    SUM(quantidade) unidades, SUM(valor_liquido) receita_liquida
    FROM fato_vendas GROUP BY ano, mes, categoria, uf""",
 'ranking_produtos': """SELECT *, ROW_NUMBER() OVER (ORDER BY receita_liquida DESC, produto_id) posicao
    FROM (SELECT produto_id, nome_produto, categoria, COUNT(*) vendas,
    SUM(quantidade) unidades, SUM(valor_liquido) receita_liquida
    FROM fato_vendas GROUP BY produto_id, nome_produto, categoria)""",
 'resumo_executivo': """SELECT COUNT(*) vendas, COUNT(DISTINCT cliente_id) clientes_ativos,
    COUNT(DISTINCT produto_id) produtos_vendidos, SUM(quantidade) unidades,
    SUM(valor_bruto) receita_bruta, SUM(valor_desconto) descontos, SUM(valor_liquido) receita_liquida,
    CAST(SUM(valor_liquido)/COUNT(*) AS DECIMAL(18,2)) ticket_medio FROM fato_vendas"""}
gold = {}
for nome, consulta in consultas.items():
    spark.sql(consulta).write.mode('overwrite').parquet(f'{GOLD}/{nome}')
    gold[nome] = spark.read.parquet(f'{GOLD}/{nome}')
    gold[nome].createOrReplaceTempView(f'gold_{nome}')
gold['resumo_executivo'].show(truncate=False)
gold['ranking_produtos'].orderBy('posicao').show(10,truncate=False)

+------+---------------+-----------------+--------+-------------+------------+---------------+------------+
|vendas|clientes_ativos|produtos_vendidos|unidades|receita_bruta|descontos   |receita_liquida|ticket_medio|
+------+---------------+-----------------+--------+-------------+------------+---------------+------------+
|995000|248750         |99500            |2985000 |1506712450.00|113066412.50|1393646037.50  |1400.65     |
+------+---------------+-----------------+--------+-------------+------------+---------------+------------+

+----------+-------------+---------+------+--------+---------------+-------+
|produto_id|nome_produto |categoria|vendas|unidades|receita_liquida|posicao|
+----------+-------------+---------+------+--------+---------------+-------+
|39213     |Produto 39213|Esporte  |10    |50      |30594.00       |1      |
|52233     |Produto 52233|Esporte  |10    |50      |30584.00       |2      |
|65253     |Produto 65253|Esporte  |10    |50      |30574.00       |3     

## 8. Reconciliação e critérios de sucesso
Com o volume padrão, esperamos **995.000 vendas válidas e 5.000 em quarentena** (1.000 por motivo injetado). Conferimos conservação das linhas, ausência de referências órfãs e igualdade exata da receita entre silver e gold, sem tolerância de ponto flutuante.

A soma das contagens e valores nas agregações deve coincidir com a fato. A quarentena é excluída dos indicadores comerciais.

In [8]:
assert v.join(sc,'cliente_id','left_anti').limit(1).count() == 0
assert v.join(sp,'produto_id','left_anti').limit(1).count() == 0
receita_silver = v.agg(F.sum('valor_liquido').alias('r')).first()['r']
for nome in ['vendas_diarias','vendas_mensais','ranking_produtos']:
    totais = gold[nome].agg(F.sum('vendas').alias('n'),F.sum('receita_liquida').alias('r')).first()
    assert totais['n'] == n_validas and totais['r'] == receita_silver, nome
resumo = gold['resumo_executivo'].first()
assert resumo['vendas'] == n_validas and resumo['receita_liquida'] == receita_silver
assert v.filter('valor_liquido < 0 OR valor_liquido IS NULL').limit(1).count() == 0
print('Todas as reconciliações passaram.')

Todas as reconciliações passaram.


## 9. Consultas e leitura após reiniciar
As views temporárias desaparecem ao encerrar a sessão; os arquivos podem ser lidos novamente por caminho. A consulta abaixo filtra partições ano/mês e permite observar `PartitionFilters` no plano físico. Evite particionar por cliente ou produto: isso criaria muitas pastas e arquivos pequenos.

O diretório de cada tabela contém vários arquivos Parquet: ele não representa um único arquivo. Evite `coalesce(1)` em dados grandes.

In [9]:
v.createOrReplaceTempView('silver_vendas')
spark.sql("SELECT canal, SUM(valor_liquido) receita FROM silver_vendas WHERE ano=2025 AND mes=1 GROUP BY canal").show()
v.filter((F.col('ano') == 2025) & (F.col('mes') == 1)).explain()
# Após reiniciar, informe o caminho impresso na seção 2:
# spark.read.parquet('/caminho/da/execucao/gold/resumo_executivo').show()

+------+-----------+
| canal|    receita|
+------+-----------+
|Online|59381884.15|
|  Loja|56407407.04|
+------+-----------+

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [venda_id#3637L,cliente_id#3638L,produto_id#3639L,data_venda#3640,quantidade#3641,preco_unitario#3642,desconto_pct#3643,canal#3644,_lote#3645,_ingerido_em#3646,_arquivo_origem#3647,valor_bruto#3648,valor_desconto#3649,valor_liquido#3650,ano#3651,mes#3652] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/lakehouse_vendas_batch/20261011T210239023684Z/s..., PartitionFilters: [isnotnull(ano#3651), isnotnull(mes#3652), (ano#3651 = 2025), (mes#3652 = 1)], PushedFilters: [], ReadSchema: struct<venda_id:bigint,cliente_id:bigint,produto_id:bigint,data_venda:date,quantidade:int,preco_u...




## 10. Auditoria do lote
O manifesto é gravado apenas após os testes. Ele indica que este processo terminou com sucesso; não substitui um commit transacional ou um catálogo com controle de concorrência. Consumidores devem selecionar somente execuções com manifesto e não ler pastas em construção.

A escrita JSON local é apropriada para este exemplo de uma máquina. Em cluster, publique metadados no armazenamento compartilhado ou em um serviço de controle de lotes.

In [10]:
import json
spark.createDataFrame(metricas,['entidade','camada','registros']).write.mode('overwrite').parquet(str(BASE/'auditoria'/'contagens'))
manifesto = {'run_id':RUN_ID,'status':'SUCESSO','formato':'parquet','modo':'batch_snapshot',
    'spark':spark.version,'clientes':N_CLIENTES,'produtos':N_PRODUTOS,'vendas_origem':N_VENDAS,
    'vendas_validas':n_validas,'quarentena':n_quarentena,'receita_liquida':str(receita_silver),
    'duracao_segundos':round(time.perf_counter()-inicio,2),'base':str(BASE)}
(BASE/'manifesto_sucesso.json').write_text(json.dumps(manifesto,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(manifesto,ensure_ascii=False,indent=2))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

{
  "run_id": "20261011T210239023684Z",
  "status": "SUCESSO",
  "formato": "parquet",
  "modo": "batch_snapshot",
  "spark": "3.5.0",
  "clientes": 250000,
  "produtos": 100000,
  "vendas_origem": 1000000,
  "vendas_validas": 995000,
  "quarentena": 5000,
  "receita_liquida": "1393646037.50",
  "duracao_segundos": 107.28,
  "base": "/home/jovyan/work/lakehouse_vendas_batch/20261011T210239023684Z"
}


## 11. Operação, evolução e exercícios
- **Batch:** agende o notebook ou extraia suas etapas para um script via cron/Airflow. Uma execução trata um snapshot completo; não usa Structured Streaming.
- **Incremental:** defina uma chave de lote, registro de arquivos processados e estratégia de reconstrução das partições afetadas. `append` isolado não impede duplicações.
- **Schema:** versionar contratos e tratar migrações explicitamente. Parquet não oferece `overwriteSchema` de Delta; reconstruir um snapshot com esquema novo exige alinhar consumidores.
- **Publicação:** a pasta por execução reduz conflitos, mas leitores precisam de um ponteiro/catalogação e protocolo de publicação; rename em object storage não garante transação de várias tabelas.
- **Segurança:** em dados reais, proteger informações pessoais, definir retenção e separar acesso à bronze e à gold.
- **Performance:** ajustar partições à capacidade, inspecionar `explain` e Spark UI; broadcast apenas quando a dimensão realmente couber na memória. O exemplo deixa essa decisão ao otimizador.
- **Falhas:** um erro de Java exige conferir `JAVA_HOME` e a versão. Uma sessão previamente configurada com extensões Delta deve ser encerrada e o kernel reiniciado antes de usar esta configuração limpa.

**Exercícios:** adicionar vendas canceladas; calcular receita por UF; simular conversões inválidas; criar um segundo lote e comparar indicadores; propor idempotência incremental sem duplicar vendas.

## 12. Referências oficiais
- [Spark 3.5.7: visão geral e requisitos](https://spark.apache.org/docs/3.5.7/)
- [Parquet e descoberta de partições](https://spark.apache.org/docs/3.5.7/sql-data-sources-parquet.html)
- [Leitura e escrita: modos de salvamento](https://spark.apache.org/docs/3.5.7/sql-data-sources-load-save-functions.html)
- [Performance do Spark SQL](https://spark.apache.org/docs/3.5.7/sql-performance-tuning.html)

A documentação entre as células explica os objetivos e limites das escolhas. Os números de receita são obtidos ao executar o notebook, não estimados.

In [ ]:
# Opcional: encerre a sessão apenas quando terminar as consultas.
# spark.stop()